# HE regression heritability/shared-environment estimators

Fits the estimator table from `notes.md` against `08_accumulate.ipynb`'s
binned, merged cross-product output (`~/grm_pheno_cov_1kg_eur/{tag}_merged.full.tsv`,
one file per `(phenotype, transform, covariate_set)`). Run `08_accumulate.ipynb`
first; this notebook never calls the tool, it only reads that output.

**Each row of a merged file is one `(class, bin)`, not one bin.** That is the
substantive change from the pre-classification version of this notebook. With
`--pair-classes` in play, every bin appears once per pair class — `other`, `FS`,
`PO`, `Dup_MZ` — plus once as `pooled`, which `merge` synthesises as the sum
over the real classes and which therefore reproduces exactly what an unclassed
run would have produced. Filtering on `bin_midpoint` alone, as this notebook
used to, feeds the same bin to the regression up to five times: it mixes PO
into the FS fit (the precise contamination the classification exists to remove),
double-counts every pair via `pooled`, and understates every standard error by
roughly 2x because `dof` and `sigma2` treat the duplicates as independent
observations. **Every fit here now names the class it is fitted on.**

Which class each estimator wants is a modelling choice, not a mechanical one:

| estimator | class | why |
|---|---|---|
| `h2_Unrel` | `other` | no classified pairs live below `a_ij = 0.02`, so `other` and `pooled` agree there |
| `h2_FS`, `b2_FS` | `FS` | the point of the exercise: a first-degree slope with PO pairs removed |
| `h2_PO`, `b2_PO` | `PO` | PO mean cross-product is a deliverable in its own right, not just something excluded |
| `h2_Ped_W25`, `h2_Ped_f` | `pooled` | these span 0.05–0.7 and are defined over all pairs in that range |
| `b2_step` | `FS` / `other` | FS band from the FS class; the HS band was never classified (KING ran at degree 1), so `other` |

`h2_Unrel`, `h2_FS` and `b2_FS` are *additionally* refitted on `pooled`, giving
the directly comparable pre-classification number alongside the classed one.

Columns used per row: `bin_midpoint` (mean `a_ij` in the bin), `full_mean`
(mean cross-product), `full_n` (pair count), `jk_se` (block-jackknife SE of the
bin mean, `NBLOCKS=50`). All regressions are **bin-level weighted least
squares** with weight `1/jk_se**2` — this pipeline never assembles a per-pair
table, so per-bin jackknife SEs stand in for pair-level bootstrap. Note
`pooled`'s `jk_se` is a real jackknife SE, not a combination of the per-class
ones: `merge` sums the delete-block accumulators across classes *before*
running the jackknife.

No bootstrap CI in this pass — WLS's own analytic SEs are reported.

## Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

# 08_accumulate.ipynb's merge output lives here.
WORK_DIR = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = (f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
              f"/07_estimators/he_regression")
os.makedirs(BUCKET_DIR, exist_ok=True)   # survives VM deletion, unlike WORK_DIR

print("WORK_DIR:  ", WORK_DIR)
print("BUCKET_DIR:", BUCKET_DIR)

## Discover merged combos

Globs for whatever `*_merged.full.tsv` files `08_accumulate.ipynb` actually
produced rather than hardcoding a phenotype/transform/covset list. Filename
convention: `{phenotype_name}__{transform}__{covariate_set}_merged.full.tsv`.

Also records which pair classes are present, so a fit whose class this run
never produced is skipped with a note instead of failing.

In [ ]:
merged_files = sorted(glob.glob(f"{WORK_DIR}/*_merged.full.tsv"))
assert merged_files, (
    f"No *_merged.full.tsv under {WORK_DIR} — run 08_accumulate.ipynb first "
    f"(and check its RUN_DIR matches this notebook's)"
)

combos = []
for path in merged_files:
    stem = os.path.basename(path)[: -len("_merged.full.tsv")]
    parts = stem.split("__", 2)
    if len(parts) != 3:
        print(f"  skipping unparseable filename: {path}")
        continue
    combos.append(tuple(parts) + (path,))

print(f"{len(combos)} phenotype/transform/covariate_set combos found")

# Which pair classes the accumulate run actually produced. An older, unclassed
# merged file has no `class` column at all; treat that as a single implicit
# `pooled` class so this notebook still reads it, rather than failing.
_probe = pd.read_csv(combos[0][3], sep="\t", nrows=5000)
HAS_CLASS = "class" in _probe.columns
AVAILABLE_CLASSES = (sorted(_probe["class"].unique()) if HAS_CLASS else ["pooled"])
print(f"class column present: {HAS_CLASS}")
print(f"classes available:    {AVAILABLE_CLASSES}")
if not HAS_CLASS:
    print("\nNOTE: unclassed output — every fit below will read as class 'pooled', "
          "which is\n      what an unclassed run means. Re-run 08_accumulate.ipynb "
          "with --pair-classes\n      to separate PO from FS.")


def read_merged(path):
    df = pd.read_csv(path, sep="\t")
    if "class" not in df.columns:
        df["class"] = "pooled"
    return df

## Estimator definitions

Ported from `notes.md`'s table and cross-checked against the legacy
`GRM-pairs/regress_y/estimator_fits.R`, which implements the same models at the
pair level (`h2_unrel`/`h2_FS` -> no-intercept slope, `h2_ped_W26` -> quadratic
no-intercept, `h2_ped_SE_factors` -> slope plus indicator offsets).

Each spec now carries a `cls`: the pair class its bins are taken from. See the
table in the header for why each is what it is. `b2_step`'s two ranges (FS
0.4–0.6, HS 0.2–0.3) match the legacy `h2_FS`/`h2_HS` ranges.

In [ ]:
# `cls` names the pair class each estimator is fitted on -- see the header table.
ESTIMATORS = [
    {"name": "h2_Unrel",   "cls": "other",  "range": (-np.inf, 0.02), "terms": ["a"],                  "intercept": False},
    {"name": "h2_FS",      "cls": "FS",     "range": (0.4, 0.6),      "terms": ["a"],                  "intercept": False},
    {"name": "h2_PO",      "cls": "PO",     "range": (0.4, 0.6),      "terms": ["a"],                  "intercept": False},
    {"name": "h2_Ped_W25", "cls": "pooled", "range": (0.05, 0.7),     "terms": ["a", "a2"],            "intercept": False},
    {"name": "h2_Ped_f",   "cls": "pooled", "range": (0.05, 0.7),     "terms": ["a", "f1", "f2", "f3"], "intercept": False},
    {"name": "b2_FS",      "cls": "FS",     "range": (0.4, 0.6),      "terms": ["a"],                  "intercept": True},
    {"name": "b2_PO",      "cls": "PO",     "range": (0.4, 0.6),      "terms": ["a"],                  "intercept": True},
]

# Refitted on `pooled` as well, to sit beside the classed fit as the directly
# comparable pre-classification number. Only the class-sensitive ones: the
# Ped estimators are already pooled, and h2_PO/b2_PO on pooled would just
# duplicate h2_FS/b2_FS under a misleading name.
POOLED_REFIT = {"h2_Unrel", "h2_FS", "b2_FS"}

B2_STEP_FS_RANGE, B2_STEP_FS_CLASS = (0.4, 0.6), "FS"
# The HS band was never classified -- KING ran at --degree 1, so 2nd-degree
# pairs are not in deg1_classified.tsv and therefore sit in `other`.
B2_STEP_HS_RANGE, B2_STEP_HS_CLASS = (0.2, 0.3), "other"


def build_design(a, terms):
    cols = []
    for t in terms:
        if t == "a":
            cols.append(a)
        elif t == "a2":
            cols.append(a ** 2)
        elif t == "f1":
            cols.append(((a >= 0.1) & (a < 0.2)).astype(float))
        elif t == "f2":
            cols.append(((a >= 0.2) & (a < 0.4)).astype(float))
        elif t == "f3":
            cols.append(((a >= 0.4) & (a < 0.6)).astype(float))
        else:
            raise ValueError(f"unknown term: {t}")
    return np.column_stack(cols)


def wls_fit(X, y, se):
    # Standard weighted least squares, weight = 1/se**2 -- the same fit
    # numpy.polyfit(..., w=1/se) produces (polyfit applies w unsquared to both
    # X and y, algebraically identical to normal equations with weight 1/se**2).
    w = 1.0 / np.asarray(se) ** 2
    XtW = X.T * w
    XtWX = XtW @ X
    beta = np.linalg.solve(XtWX, XtW @ y)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    if dof <= 0:
        return beta, np.full_like(beta, np.nan), dof
    sigma2 = np.sum(w * resid ** 2) / dof
    cov = sigma2 * np.linalg.inv(XtWX)
    return beta, np.sqrt(np.diag(cov)), dof


def usable_bins(df, lo, hi, cls):
    """Bins in [lo, hi] for ONE pair class.

    The class filter is not optional. Without it each bin is returned once per
    class plus once as `pooled`, which mixes classes into a single regression
    and inflates dof.
    """
    sub = df[df["class"] == cls]
    sub = sub[(sub["bin_midpoint"] >= lo) & (sub["bin_midpoint"] <= hi)]
    sub = sub[(sub["full_n"] > 0) & sub["jk_se"].notna() & (sub["jk_se"] > 0)]
    assert sub["bin_index"].is_unique, (
        f"class {cls!r} has repeated bin_index values in [{lo}, {hi}] -- the "
        f"merged file is malformed"
    )
    return sub


def _empty(name, cls, n_bins=0):
    return dict(estimator=name, fit_class=cls, n_bins=n_bins,
                n_pairs=0, point_estimate=np.nan, se=np.nan, dof=0)


def fit_estimator(df, spec, cls):
    lo, hi = spec["range"]
    sub = usable_bins(df, lo, hi, cls)
    a = sub["bin_midpoint"].to_numpy()
    y = sub["full_mean"].to_numpy()
    se = sub["jk_se"].to_numpy()

    X = build_design(a, spec["terms"])
    if spec["intercept"]:
        X = np.column_stack([X, np.ones(len(a))])

    n_bins = len(sub)
    n_pairs = int(sub["full_n"].sum())
    if n_bins <= X.shape[1]:
        return _empty(spec["name"], cls, n_bins)

    beta, beta_se, dof = wls_fit(X, y, se)

    # b2_FS/b2_PO: b2 = 2 * intercept (last column). Every other estimator
    # reports the slope on `a`, which is always column 0.
    if spec["name"].startswith("b2_"):
        point, point_se = 2 * beta[-1], 2 * beta_se[-1]
    else:
        point, point_se = beta[0], beta_se[0]

    return dict(estimator=spec["name"], fit_class=cls, n_bins=n_bins,
                n_pairs=n_pairs, point_estimate=point, se=point_se, dof=dof)


def fit_b2_step(df, fs_cls=B2_STEP_FS_CLASS, hs_cls=B2_STEP_HS_CLASS):
    fs = usable_bins(df, *B2_STEP_FS_RANGE, fs_cls)
    hs = usable_bins(df, *B2_STEP_HS_RANGE, hs_cls)
    label = f"{fs_cls}/{hs_cls}"
    if len(fs) <= 2 or len(hs) <= 2:
        return _empty("b2_step", label, len(fs) + len(hs))

    X_fs = np.column_stack([fs["bin_midpoint"].to_numpy(), np.ones(len(fs))])
    beta_fs, se_fs, dof_fs = wls_fit(X_fs, fs["full_mean"].to_numpy(), fs["jk_se"].to_numpy())

    X_hs = np.column_stack([hs["bin_midpoint"].to_numpy(), np.ones(len(hs))])
    beta_hs, se_hs, dof_hs = wls_fit(X_hs, hs["full_mean"].to_numpy(), hs["jk_se"].to_numpy())

    point = 4 * (beta_fs[-1] - beta_hs[-1])
    # independent fits on disjoint bin ranges -- errors add in quadrature
    point_se = 4 * np.sqrt(se_fs[-1] ** 2 + se_hs[-1] ** 2)
    return dict(estimator="b2_step", fit_class=label,
                n_bins=len(fs) + len(hs),
                n_pairs=int(fs["full_n"].sum() + hs["full_n"].sum()),
                point_estimate=point, se=point_se, dof=min(dof_fs, dof_hs))


# (spec, class) pairs to fit, skipping classes this run didn't produce.
JOBS, _skipped = [], []
for spec in ESTIMATORS:
    for cls in dict.fromkeys([spec["cls"]] + (["pooled"] if spec["name"] in POOLED_REFIT else [])):
        if cls in AVAILABLE_CLASSES:
            JOBS.append((spec, cls))
        else:
            _skipped.append(f"{spec['name']}[{cls}]")

print(f"{len(JOBS)} (estimator, class) fits per combo:")
for spec, cls in JOBS:
    print(f"  {spec['name']:<12} on {cls}")
B2_STEP_OK = B2_STEP_FS_CLASS in AVAILABLE_CLASSES and B2_STEP_HS_CLASS in AVAILABLE_CLASSES
print(f"  {'b2_step':<12} on {B2_STEP_FS_CLASS}/{B2_STEP_HS_CLASS}"
      f"{'' if B2_STEP_OK else '   -- SKIPPED, class missing'}")
if _skipped:
    print(f"\nskipped (class absent from this run): {', '.join(_skipped)}")

## Fit every estimator for every combo

In [ ]:
rows = []
for phenotype_name, transform, covariate_set, path in combos:
    df = read_merged(path)
    meta = dict(phenotype_name=phenotype_name, transform=transform,
                covariate_set=covariate_set)
    for spec, cls in JOBS:
        rows.append({**fit_estimator(df, spec, cls), **meta})
    if B2_STEP_OK:
        rows.append({**fit_b2_step(df), **meta})

results = pd.DataFrame(rows)[
    ["phenotype_name", "transform", "covariate_set", "estimator", "fit_class",
     "n_bins", "n_pairs", "point_estimate", "se", "dof"]
]

out_tsv = f"{BUCKET_DIR}/he_estimator_results.tsv"
results.to_csv(out_tsv, sep="\t", index=False)
print(f"Wrote {len(results)} rows to {out_tsv}")
print(f"\n{results['estimator'].nunique()} estimators x "
      f"{results['fit_class'].nunique()} classes over {len(combos)} combos")
results.head(20)

## PO vs FS — the result the classification was built for

PO and FS pairs both sit at `a_ij ~= 0.5`, so before classification they shared
a bin and their cross-products were averaged together. Split, they answer
different questions: FS pairs share dominance (`d_ij = 0.25`) and, typically,
a rearing environment; PO pairs share neither (`d_ij = 0`, and parent/offspring
are usually not raised together). So `b2_FS > b2_PO` is the expected direction,
and `h2_FS` above `h2_PO` is the signature of shared-environment or dominance
contribution loading onto the FS slope.

The `pooled` row is the pre-classification number: it should sit between the
two, weighted by their pair counts. If it does not, the classification and the
binning disagree about something.

In [ ]:
deg1 = results[results["estimator"].isin(["h2_FS", "h2_PO", "b2_FS", "b2_PO"])]
pivot = deg1.pivot_table(
    index=["phenotype_name", "transform", "covariate_set"],
    columns=["estimator", "fit_class"], values="point_estimate",
)
print("pairs per class entering these fits (first combo):")
print(deg1.groupby(["estimator", "fit_class"])["n_pairs"].first().to_string())
print("\nFS vs PO point estimates:")
pivot.head(20)

### Sanity checks

`n_bins` is the number of bins entering each fit. Before the class filter was
added, these came out ~5x larger because every bin appeared once per class plus
once as `pooled` — so an implausibly large `n_bins` is the tell that a fit is
reading more than one class.

`h2_Unrel` on `other` and on `pooled` should be **identical**: no pair below
`a_ij = 0.02` is classified, so the two classes hold the same bins there. A
difference means classified pairs are landing in the unrelated region, which
would point at a bad `deg1_classified.tsv`.

In [ ]:
print("n_bins per estimator (max over combos) — watch for ~5x inflation:")
print(results.groupby(["estimator", "fit_class"])["n_bins"].max().to_string())

unrel = results[results["estimator"] == "h2_Unrel"]
if unrel["fit_class"].nunique() > 1:
    w = unrel.pivot_table(index=["phenotype_name", "transform", "covariate_set"],
                          columns="fit_class", values="point_estimate")
    if {"other", "pooled"} <= set(w.columns):
        d = (w["other"] - w["pooled"]).abs()
        print(f"\nh2_Unrel other vs pooled: max |difference| = {d.max():.3e}")
        assert d.max() < 1e-9 or np.isnan(d.max()), (
            "h2_Unrel differs between 'other' and 'pooled' — classified pairs are "
            "present below a_ij = 0.02, so deg1_classified.tsv is suspect"
        )
        print("identical, as expected")

print("\ntop h2_Unrel estimates:")
results[(results["estimator"] == "h2_Unrel") & (results["fit_class"] == "other")] \
    .sort_values("point_estimate", ascending=False) \
    [["phenotype_name", "transform", "covariate_set", "point_estimate", "se", "n_bins"]] \
    .head(20)

## Forest plot per phenotype/transform

One plot per `(phenotype_name, transform)`: point estimate ±1 SE per
`(estimator, class)`, coloured by covariate set. Each row label carries its
class, so the FS/PO contrast and the classed-vs-`pooled` comparison are both
readable off the same axis. Estimates whose class had too few bins come through
as NaN and are simply absent.

In [ ]:
PLOTS_DIR = f"{BUCKET_DIR}/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# Row order follows ESTIMATORS, with each estimator's classes grouped together
# and b2_step last -- derived from `results` so skipped fits leave no empty row.
_spec_order = [e["name"] for e in ESTIMATORS] + ["b2_step"]
present = results[["estimator", "fit_class"]].drop_duplicates()
row_keys = sorted(
    (tuple(r) for r in present.to_numpy()),
    key=lambda kc: (_spec_order.index(kc[0]) if kc[0] in _spec_order else 99, kc[1]),
)
row_labels = [f"{name}  [{cls}]" for name, cls in row_keys]
cmap = plt.get_cmap("viridis")

saved_plots = []
for (phenotype_name, transform), g in results.groupby(["phenotype_name", "transform"]):
    covsets = sorted(g["covariate_set"].unique())
    covset_color = {cs: cmap(i / max(1, len(covsets) - 1)) for i, cs in enumerate(covsets)}

    fig, ax = plt.subplots(figsize=(7, 0.42 * len(row_keys) + 1.6))
    for j, cs in enumerate(covsets):
        d = (g[g["covariate_set"] == cs]
             .set_index(["estimator", "fit_class"])
             .reindex(row_keys))
        y_pos = np.arange(len(row_keys)) + (j - (len(covsets) - 1) / 2) * 0.16
        ax.errorbar(d["point_estimate"], y_pos, xerr=d["se"], fmt="o", ms=4,
                    color=covset_color[cs], label=cs)

    ax.axvline(0, color="grey", lw=0.5)
    # separate the estimator groups so the class comparisons read as blocks
    for i in range(1, len(row_keys)):
        if row_keys[i][0] != row_keys[i - 1][0]:
            ax.axhline(i - 0.5, color="0.9", lw=0.8)
    ax.set_yticks(np.arange(len(row_keys)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("estimate")
    ax.set_title(f"{phenotype_name} ({transform})")
    ax.legend(title="covariate set", fontsize=8)
    plt.tight_layout()

    out_png = f"{PLOTS_DIR}/{phenotype_name}__{transform}__estimators.png"
    fig.savefig(out_png, dpi=150)
    plt.show()
    plt.close(fig)
    saved_plots.append(out_png)

print(f"{len(saved_plots)} plots saved to {PLOTS_DIR}")

## Copy notebook to bucket

In [ ]:
import subprocess

_nb = os.path.expanduser("~/repos/AOU-covariance/notebooks/09_estimators.ipynb")
_gs = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
       f"{PROJECT_DIR}/{RUN_DIR}/notebooks/09_estimators.ipynb")
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")